# Pilot analysis: flatness vs OOD generalization

Does a distilled student that beats its teacher on CIFAR-10-C also sit in a
flatter minimum? And does the pattern hold in both fp32 and AMP?

Input is `results/pilot_summary.csv` from
`python -m src.evaluate --aggregate`, after training, `evaluate --all`, and
`measure_geometry --all`. Relevant columns: `id_acc`, `ood_acc_mean`,
`mce_vs_baseline`, `adaptive_sharpness`, `hessian_trace`,
`hessian_top_eigenvalue`, plus `mode`, `precision`, `width_mult`, `seed`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv = Path('../results/pilot_summary.csv')
assert csv.exists(), f'{csv} not found; run the pilot and aggregate first'
df = pd.read_csv(csv)
df['label'] = np.where(df['mode'] == 'teacher', 'teacher',
                       'student w' + df['width_mult'].astype(str))
df = df.sort_values(['precision', 'mode', 'width_mult', 'seed']).reset_index(drop=True)
df[['run_name', 'mode', 'precision', 'width_mult', 'seed', 'id_acc',
    'ood_acc_mean', 'mce_vs_baseline', 'adaptive_sharpness',
    'hessian_trace', 'hessian_top_eigenvalue']]

## Mean and std over seeds

In [ ]:
metrics = ['id_acc', 'ood_acc_mean', 'mce_vs_baseline',
           'adaptive_sharpness', 'hessian_trace', 'hessian_top_eigenvalue']
df.groupby(['precision', 'label'])[metrics].agg(['mean', 'std']).round(4)

## Student vs its own teacher

Each student is compared to the teacher of the same precision and seed.

In [ ]:
teachers = df[df['mode'] == 'teacher'].set_index(['precision', 'seed'])[metrics]
rows = []
for _, s in df[df['mode'] == 'student'].iterrows():
    key = (s['precision'], s['seed'])
    if key not in teachers.index:
        continue
    t = teachers.loc[key]
    rows.append({
        'precision': s['precision'], 'seed': s['seed'], 'student': s['label'],
        'd_id_acc': s['id_acc'] - t['id_acc'],
        'd_ood_acc': s['ood_acc_mean'] - t['ood_acc_mean'],
        'beats_teacher_ood': s['ood_acc_mean'] > t['ood_acc_mean'],
        'd_sharpness': s['adaptive_sharpness'] - t['adaptive_sharpness'],
        'flatter_than_teacher': s['adaptive_sharpness'] < t['adaptive_sharpness'],
        'd_hessian_trace': s['hessian_trace'] - t['hessian_trace'],
    })
cmp = pd.DataFrame(rows)
cmp

In [ ]:
pd.crosstab(cmp['beats_teacher_ood'], cmp['flatter_than_teacher'],
            rownames=['beats teacher OOD'], colnames=['flatter than teacher'])

## Sharpness vs OOD accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, prec in zip(axes, ['fp32', 'amp']):
    sub = df[df['precision'] == prec]
    if sub.empty:
        ax.set_title(f'{prec} (no runs)')
        continue
    for lab, g in sub.groupby('label'):
        ax.scatter(g['adaptive_sharpness'], g['ood_acc_mean'], s=60, label=lab)
    ax.set_xlabel('adaptive sharpness (lower is flatter)')
    ax.set_title(prec)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('CIFAR-10-C mean accuracy')
axes[0].legend()
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for lab, g in df.groupby('label'):
    ax.scatter(g['hessian_trace'], g['ood_acc_mean'], s=60, label=lab)
ax.set_xlabel('Hessian trace (lower is flatter)')
ax.set_ylabel('CIFAR-10-C mean accuracy')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

## fp32 vs AMP

In [ ]:
if df['precision'].nunique() < 2:
    print('Only one precision present. Train the AMP configs to compare.')
else:
    piv = (df.groupby(['label', 'precision'])[['ood_acc_mean', 'adaptive_sharpness']]
             .mean().unstack('precision'))
    print(piv.round(4))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip(axes, ['ood_acc_mean', 'adaptive_sharpness']):
        piv[m].plot(kind='bar', ax=ax)
        ax.set_title(m)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=20)
        ax.grid(alpha=0.3)
    fig.tight_layout()

## What to look for

The hypothesis is supported to the extent that:

1. students that beat the teacher on CIFAR-10-C fall mostly in the
   flatter-than-teacher column of the contingency table above;
2. OOD accuracy drops as sharpness rises across models, not just with width;
3. the same holds in fp32 and AMP.

With 3 seeds this is about effect direction and per-seed consistency, not
significance.